# FullVersion：种植、采矿、钓鱼的共享资源联立优化

本 notebook 将三个模块的内部约束加入同一个 Gurobi model，并共享 500g 初始现金、每日 900 分钟和 270 体力。锄头、水壶和镐子共用黑匠提交与服务容量；钓鱼竿是商店购买。目标为最大化 Day 112 日末共享现金。

## 已确认口径

模型使用唯一共享现金递推：$B_t=B_{t-1}-[C_t^P+C_t^M+C_t^F]+[R_t^P+R_t^M+R_t^F]$，其中 $R_t^F=\sum_{i\in\Omega_t}v_ix_i$。不维护矿石、煤炭或金属锭库存；采矿产出直接变现。完整公式见 `全模块联立模型_变量与约束说明.md`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'planting').exists():
    PROJECT_ROOT = next(p for p in Path.cwd().resolve().parents if (p / 'planting').exists())

ACTIVE_MODULES = ['planting', 'mining', 'fishing']
HORIZON_DAYS = 112
INITIAL_CASH = 500.0
DAILY_TIME = 900.0
DAILY_ENERGY = 270.0
ENFORCE_ONE_BLACKSMITH_SUBMISSION = True
ENFORCE_ONE_BLACKSMITH_SERVICE = True
TIME_LIMIT_SECONDS = 300
MIP_GAP = 0.001

COMBINE_DIR = PROJECT_ROOT / 'CombineAndCompare'
if str(COMBINE_DIR) not in sys.path:
    sys.path.insert(0, str(COMBINE_DIR))
from full_version_solver import solve_full_version

print('Active modules:', ACTIVE_MODULES)
print('Shared daily resources:', DAILY_TIME, 'minutes;', DAILY_ENERGY, 'energy')


## Shared constraints to be added

三个 adapter 分别加入种植、采矿和钓鱼的内部约束，但不创建独立现金或独立总资源上限。`module_contract.add_full_version_constraints` 只添加一条共享现金路径、共享时间/体力预算，以及三把黑匠工具的提交和服务容量约束。求解和导出可能需要数分钟。

In [ ]:
result = solve_full_version(PROJECT_ROOT, time_limit=TIME_LIMIT_SECONDS, mip_gap=MIP_GAP)
summary = result['summary']
print(f"Day 112 shared cash: {summary['terminal_cash_g']:,.2f}g")
print(f"MIP gap: {summary['mip_gap']:.4%}")
print('Output directory:', result['output_dir'])
display(result['daily'])
display(result['daily'][['day', 'cash_start_g', 'planting_revenue_g', 'mining_revenue_g', 'fishing_revenue_g', 'total_time', 'total_energy', 'cash_end_g']])
